In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import requests as req
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score,roc_auc_score
from datetime import datetime,timedelta

In [4]:
API_KEY = 'f38bb78f99b35bcb38646b2b9d13be59'
BASE_URL = 'https://api.openweathermap.org/data/2.5/'

FETCH CURRENT DATA

In [5]:
def get_current_weather(city):
    url = f"{BASE_URL}weather?q={city}&appid={API_KEY}&units=metric"

    resposne = req.get(url)
    data = resposne.json
    return{
        'city' : data['name'],
        'current_temp' : round(data['main']['temp']),
        'feels_like' : round(data['main']['feels_like']),
        'temp_min' : round(data['main']['temp_min']),
        'temp_max' : round(data['main']['temp_max']),
        'humidity' : round(data['main']['humidity']),
        'description' : data['weather'][0]['description'],
        'country' : data['sys']['country'],
    }


In [6]:
def read_historical_data(filename):
    df = pd.read_csv(filename)
    df.dropna()
    df = df.drop_duplicates

    return df

In [9]:
def prepare_data(data):
    encode = LabelEncoder(data)
    data['WindGustDir'] = encode.fit_transform(data['WindGustDir'])
    data['RainTomorrow'] = encode.fit_transform(data['RainTomorrow'])

    X = data['MinTemp','MaxTemp','WindGustDir','WindGustSpeed','Humidity','Pressure,Temp']
    y = data['RainTomorrow']


    return X,y,encode

In [13]:
def train_model(X,y):

    X_train,y_train,X_test,y_test = train_test_split(X,y, test_size=0.2, random_state=42)

    model  = RandomForestClassifier(n_estimators=100,random_state=42)

    model.fit(X_train,y_train)

    y_pred = model.predict(X_test,y_test)

    print(mean_squared_error(y_test,y_pred))
    print(mean_absolute_error(y_test,y_pred))

    print(r2_score(y_test,y_pred))

    return model


In [14]:
def prepare_forecasting(data,feature):

    X,y = [], []

    for i in range(len(data) - 1):
        X.append(data[feature].iloc[i])
        y.append(data[feature].iloc[i] + 1)

    X = np.array(X).reshape(-1,1)
    y = np.array(y)

    return X,y

In [15]:
def train_regressor_model(X,y):
    model = RandomForestRegressor(n_estimators=100,random_state=42)
    model.fit(X,y)

    return model